# Lab 3: Extracting Names, Dates and Amounts with spaCy

**Time:** about 75 minutes

Branch and claims officers write short notes on every case:

> *Member Neema Mushi (104837265) visited Ilala branch on 12/03/2025 about an unpaid maternity benefit of 1,450,000/=.*

To a person, that note is full of facts: who, which member number, where, when, how much. To a database it's one
long string. **Named entity recognition (NER)** turns the string into fields you can search, join and count. With
it you could ask *"show me every note that mentions this member"* or *"total amounts mentioned in complaints about
Temeke this quarter"*.

### What is an "entity"?

An **entity** is a specific thing mentioned in text that you'd want to pull out as a value: a person's name, a date,
an amount of money, an organisation. NER finds each one and says two things about it:

- **where** it is (its **span**: the start and end character positions in the text);
- **what type** it is (its **label**: PERSON, DATE, MONEY ...).

In the note above, "Neema Mushi" is a PERSON from character 7 to 18, "12/03/2025" is a DATE, and so on.

You'll start with **spaCy**, a popular Python library for NLP, and its ready-made English model. You'll measure exactly
how well it does on the Fund's notes, and then teach it what it doesn't know: Tanzanian names, member numbers, claim
references, the `/=` way of writing shillings, and notes written in Swahili.

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **NER** | named entity recognition: finding names, dates, amounts and other specific values in text |
| **entity** | one such value found in the text, e.g. "Ilala" as a BRANCH |
| **span** | a stretch of text, given by its start and end character positions |
| **label / type** | what kind of entity it is: PERSON, DATE, MONEY ... |
| **answer key (gold)** | the entities a person marked up by hand, used to check the model |
| **spaCy pipeline** | the chain of steps spaCy runs on a text: split into tokens, tag, find entities ... |
| **pipeline component** | one step in that chain; you can add your own |
| **true positive** | an entity the model found that is in the answer key (right) |
| **false positive** | an entity the model found that isn't in the answer key (wrong) |
| **false negative** | an entity in the answer key the model didn't find (missed) |
| **regular expression** | a pattern for finding text, e.g. "CLM- then 4 digits" |
| **lookbehind / lookahead** | parts of a pattern that check the text *before* or *after* a match, without including it |
| **normalise** | turn different ways of writing the same value into one standard form |
| **redaction** | replacing personal details with placeholders before text is shared |

## Where this lab fits

**What you've done so far.** In Lab 1 you classified the sentiment of whole messages and found their themes. In Lab 2 you
fine-tuned a transformer to route each message to a team. Both labs gave **one answer per message**.

**What you'll do now.** You'll pull out the individual **facts inside** a message: the member's name and number, the
branch, the employer, the date, the amount, the claim reference. You'll measure spaCy's ready-made English model against
a hand-made answer key, improve it with rules that carry local knowledge, turn 900 free-text notes into a table, and use
the same extraction to redact personal details.

**Why it matters.** Officers' notes hold information the Fund's systems don't: what the member said, which claim, how
much. Once that's in a table, it can be joined to the member register, counted by branch and searched by member. And
redaction is the first safeguard before any text leaves the Fund, a question that comes up straight away on Day 7.

**What happens next.** This completes Day 6. Day 7 moves to large language models, run on your own machine so that
member data stays under the Fund's control.

### Theory: three kinds of NLP task

Today's three labs cover the three basic shapes of NLP task, and almost every text problem is one of them or a mix:

| Shape | Output | Example | Lab |
|---|---|---|---|
| **Classify a message** | one label per message | sentiment; complaint category | 1 and 2 |
| **Label pieces of a message** | a label for each span of text | "Neema Mushi" = PERSON, "1,450,000/=" = MONEY | 3 |
| **Discover structure** | groups or themes, no labels | complaint topics | 1 |

Labelling pieces is harder to measure than classifying a message, because a prediction has to get both the **boundaries**
(where the entity starts and ends) and the **type** right. That's why Section 3 builds its own measuring function before
anything is improved.

## Imports and the standard model

- `spacy`: the NLP library; `displacy` draws found entities as coloured highlights;
- `Language` and `filter_spans`: used to add your own step to spaCy's pipeline (Section 4);
- `re`: regular expressions, for the rules;
- `json`: the answer key is stored as JSON text inside the CSV file;
- `HTML` and `display`: show displacy's drawing inside the notebook.

`spacy.load("en_core_web_sm")` loads spaCy's small English model, which was downloaded by the setup cell in Lab 1.
`nlp` is now a function: give it text and it returns a processed document. `pipe_names` lists the steps it runs.

In [ ]:
import json
import re                                   # regular expressions: patterns for finding text
import warnings

import pandas as pd
import spacy
from spacy import displacy                  # draws entities as coloured highlights
from spacy.language import Language
from spacy.util import filter_spans
from IPython.display import HTML, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 170)

# Load spaCy's small English model (downloaded by the Lab 1 setup cell).
nlp = spacy.load("en_core_web_sm")
print("spaCy", spacy.__version__, "| pipeline:", nlp.pipe_names)

> **Where we are.** spaCy's small English model is loaded as `nlp`. Before running it, look at the notes and the answer
> key a person made, because everything later is measured against it.

## 1. The case notes and their answer key

Each note comes with the entities a person marked up by hand: the exact start and end character of each one, and its
type. That's what lets you *measure* extraction, not just eyeball it.

The answer key is stored in the `entities` column as **JSON** text, a standard way of writing lists and dictionaries as
text. `json.loads` turns that text back into a Python list of dictionaries, one per entity. The table at the end shows
the answer key for the first note.

In [ ]:
notes = pd.read_csv("case_notes.csv")

# The answer key is stored as JSON text. json.loads turns it into a Python list of dictionaries.
notes["gold"] = notes["entities"].map(json.loads)

print(len(notes), "notes")
print(notes["text"].iloc[0])
pd.DataFrame(notes["gold"].iloc[0])

Count how many entities of each type there are, across all the notes' answer keys. The two loops go through every note,
then every entity in that note.

In [ ]:
# Count the entity types across all the answer keys.
all_labels = []
for gold in notes["gold"]:
    for entity in gold:
        all_labels.append(entity["label"])

pd.Series(all_labels).value_counts()

Seven entity types. Four of them (PERSON, ORG, DATE, MONEY) exist in spaCy's English model. Three (MEMBER_NO,
CLAIM_REF, BRANCH) are specific to the Fund, and no off-the-shelf model knows them.

> **Where we are.** There are seven entity types in the answer key: four the standard model knows (PERSON, ORG, DATE,
> MONEY) and three that are specific to the Fund (MEMBER_NO, CLAIM_REF, BRANCH). Now see what the model does on its own.

### Theory: how a statistical NER model decides

spaCy's model learned to find entities from many thousands of example sentences, mostly English news and web pages,
where every entity had been marked by a person. For each token it weighs clues such as:

- the word itself, and whether it starts with a capital letter;
- its **shape**: `Xxxxx` for *Neema*, `dddd` for *2025*, `dd/dd/dddd` for a date;
- the words around it: *"Mr"* before a word suggests a PERSON; *"in"* before a capitalised word suggests a place.

It's a **statistical** model: it makes its best guess from patterns it saw in training. That's powerful for entities
that look like its training data, and unreliable for anything that doesn't: Tanzanian names it never saw, nine-digit
member numbers (which look like dates or quantities), and shilling amounts written as `1,450,000/=`.

## 2. What the standard model does

Run the model on one note. The result, `doc`, holds the text split into tokens plus everything the model found.
`doc.ents` is the list of entities. For each, print its text, its label, and its character positions.
(`!r` in the f-string prints the text with quotes around it, so you can see any spaces at the edges.)

In [ ]:
# Run the model on one note. doc.ents is the list of entities it found.
doc = nlp(notes["text"].iloc[1])

for ent in doc.ents:
    print(f"{ent.text!r:35} {ent.label_:10} characters {ent.start_char}-{ent.end_char}")

A picture is easier to read. `displacy.render` highlights each entity in the text, with a colour per type.

In [ ]:
# The same, drawn as coloured highlights. (render() returns HTML; display() shows it in the notebook.)
display(HTML(displacy.render(doc, style="ent", jupyter=False)))

Look closely at the colours. Typical mistakes on these notes:

- member numbers tagged as **DATE** (nine digits look like a date to a model trained on news articles),
- branch names like *Kinondoni* tagged as **PERSON**,
- *TZS 739,000* tagged as an **ORG**, and *1,450,000/=* not recognised as money at all,
- claim references like *CLM-2024-331711* split into pieces and called a DATE or a PRODUCT.

None of this is surprising. `en_core_web_sm` learned from English news and web text, where none of these patterns
are common. Guessing isn't enough, though. You need numbers.

> **Where we are.** You've seen the standard model make plausible but wrong guesses on the Fund's notes. Before fixing
> anything, measure how wrong, so every improvement can be checked.

## 3. Measuring extraction

The standard measure is **precision, recall and F1 per entity type**, counting an entity as correct only when both its
exact span and its type match the answer key. Strict, but fair: a member number with one digit missing is wrong.

Each entity is written as a triple: (start character, end character, type). Then comparing predictions with the answer
key is comparing two **sets** of triples, and Python's set operations do the work:

- `predicted & truth`: in both, so **right**;
- `predicted - truth`: predicted but not in the key, so **wrong**;
- `truth - predicted`: in the key but not predicted, so **missed**.

In [ ]:
text = notes["text"].iloc[1]
gold = notes["gold"].iloc[1]

# The answer key as a set of triples.
truth = set()
for g in gold:
    truth.add((g["start"], g["end"], g["label"]))

# The model's entities as a set of triples.
predicted = set()
for ent in nlp(text).ents:
    predicted.add((ent.start_char, ent.end_char, ent.label_))

print("right (in both):         ", predicted & truth)
print("wrong (predicted only):  ", predicted - truth)
print("missed (answer key only):", truth - predicted)

Do that for every note and count, per entity type: right (**true positives**), wrong (**false positives**) and missed
(**false negatives**). Then:

- precision = right / (right + wrong): of what it found, the share that was correct;
- recall = right / (right + missed): of what was there, the share it found;
- F1 = the balance of the two (2 × precision × recall / (precision + recall)).

The function below does this for any spaCy pipeline, so you can measure each improvement the same way. Two details:

- `zip(notes["text"], notes["gold"])` walks through the texts and their answer keys side by side;
- `a / b if b > 0 else 0.0` means "divide, unless the bottom is zero, in which case use 0" (a type with no predictions
  would otherwise cause a division-by-zero error).

In [ ]:
LABELS = ["PERSON", "MEMBER_NO", "CLAIM_REF", "BRANCH", "ORG", "DATE", "MONEY"]


def evaluate(pipeline):
    """Precision, recall and F1 per entity type over all notes. Span and type must both match exactly."""
    # Start every count at zero.
    right = {}
    wrong = {}
    missed = {}
    for label in LABELS:
        right[label] = 0
        wrong[label] = 0
        missed[label] = 0

    for text, gold in zip(notes["text"], notes["gold"]):
        # The answer key for this note.
        truth = set()
        for g in gold:
            truth.add((g["start"], g["end"], g["label"]))

        # The pipeline's entities for this note.
        predicted = set()
        for ent in pipeline(text).ents:
            if ent.label_ in LABELS:              # ignore types we don't use (e.g. GPE, CARDINAL)
                predicted.add((ent.start_char, ent.end_char, ent.label_))

        # Count right and wrong predictions, and missed answers.
        for start, end, label in predicted:
            if (start, end, label) in truth:
                right[label] += 1
            else:
                wrong[label] += 1
        for start, end, label in truth:
            if (start, end, label) not in predicted:
                missed[label] += 1

    # Turn the counts into precision, recall and F1 for each type.
    rows = []
    for label in LABELS:
        precision = right[label] / (right[label] + wrong[label]) if right[label] + wrong[label] > 0 else 0.0
        recall = right[label] / (right[label] + missed[label]) if right[label] + missed[label] > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
        rows.append({"entity": label, "precision": precision, "recall": recall, "f1": f1})
    return pd.DataFrame(rows).set_index("entity")

Measure the standard model on all the notes:

In [ ]:
baseline = evaluate(nlp)
baseline.round(3)

MEMBER_NO, CLAIM_REF and BRANCH score zero, because the model doesn't know those types exist. Even the types it does
know score modestly on these notes. If you plugged this model straight into a system, you'd get a database full of
confident mistakes.

You have two ways to improve it:

1. **Train** the model on annotated notes. It's the right choice for fuzzy entities that follow no fixed pattern,
   but it needs hundreds of labelled examples and some care.
2. **Add rules** for entities that *do* follow a fixed pattern. Claim references, member numbers, shilling amounts
   and dates all do, and branch names come from a known list.

In practice you usually combine the two. Start with rules, because they're quick, transparent and easy to audit.

> **Where we are.** You have a measuring function, `evaluate`, and a baseline: the three Fund-specific types score zero
> and the others are modest. The quickest gains are for entities that follow a fixed pattern.

### Theory: statistical models and rules, working together

| | Statistical model | Rules (regular expressions, lists) |
|---|---|---|
| Good for | entities with no fixed format: names, organisations | entities with a fixed format: IDs, references, amounts, dates |
| Needs | many annotated examples to learn from | someone who knows the formats |
| Explaining a result | hard | easy: *this pattern matched* |
| When writing habits change | adapts somewhat | breaks quietly |

The strongest systems combine both. The approach here is a **hybrid pipeline**: let the model make its guesses, then
let rules correct it wherever a fixed pattern applies, because for those patterns a rule is far more reliable. The model
still handles everything the rules don't cover.

## 4. Rules for what follows a pattern

A rule is a **regular expression**: a pattern for text. `re.findall(pattern, text)` returns every piece of text that
matches. Try three on their own first. The symbols:

| Symbol | Means |
|---|---|
| `\d` | any digit |
| `{4}` | exactly 4 of the previous thing |
| ` ?` | an optional space (the `?` means "zero or one of the previous thing") |
| `[\d,]*` | any number of digits or commas (`*` means "zero or more") |
| `/=` | the literal characters `/=` |

In [ ]:
note = "Member Neema Mushi (104 837 265) visited Ilala branch about claim CLM-2024-331711 of 1,450,000/=."

# \d = a digit, {4} = exactly four of them. So this finds CLM- + 4 digits + - + 6 digits.
print(re.findall(r"CLM-\d{4}-\d{6}", note))

# Three groups of three digits, with an optional space (" ?") between them.
print(re.findall(r"\d{3} ?\d{3} ?\d{3}", note))

# Digits and commas ([\d,]*), then "/=": the Tanzanian way of writing shillings.
print(re.findall(r"\d[\d,]*/=", note))

Now the full list of rules, one (type, pattern) pair per line. A few more symbols appear:

| Symbol | Means | Used for |
|---|---|---|
| `a\|b` | a or b | different ways of writing the same thing |
| `(?: ... )` | a group, so `\|` applies only inside it | `(?:TZS\|Tsh\|shs)` |
| `\s` | any space character | `TZS 739,000` |
| `\b` | a word boundary | so *Moshi* doesn't match inside a longer word |
| `(?<![\d-])` | **not** preceded by a digit or a dash | a member number isn't the end of a longer number |
| `(?!\d)` | **not** followed by a digit | ... or the start of one |
| `(?= branch)` | followed by " branch" (not included in the match) | *Ilala* only when it's a branch |
| `(?<=tawi la )` | preceded by "tawi la " (Swahili for "branch of") | *tawi la Ilala* |

`"|".join(BRANCHES)` turns the list of branch names into one pattern meaning "any one of these".

In [ ]:
MONTHS = "January|February|March|April|May|June|July|August|September|October|November|December"
BRANCHES = ["Ilala", "Kinondoni", "Temeke", "Ubungo", "Arusha", "Mwanza", "Dodoma", "Mbeya", "Morogoro", "Tanga", "Moshi",
            "Iringa", "Tabora", "Bukoba", "Musoma", "Shinyanga", "Songea", "Mtwara"]
BRANCH_NAMES = "|".join(BRANCHES)          # "Ilala|Kinondoni|..." means "any one of these"

RULES = [
    # claim references: CLM-2024-331711
    ("CLAIM_REF", r"CLM-\d{4}-\d{6}"),
    # member numbers: 9 digits, spaces allowed, not part of a longer number
    ("MEMBER_NO", r"(?<![\d-])\d{3} ?\d{3} ?\d{3}(?!\d)"),
    # money: TZS 739,000 / Tsh 2.38m / shs 50,000 / 1,450,000/=
    ("MONEY", r"(?:TZS|Tsh|shs)\s?\d[\d,]*(?:\.\d+)?m?|\d[\d,]*/="),
    # dates: 12/03/2025, 2025-03-12, 5 March 2025, 05-Oct-23
    ("DATE", r"\d{2}/\d{2}/\d{4}|\d{4}-\d{2}-\d{2}|\d{1,2} (?:" + MONTHS + r") \d{4}|\d{2}-[A-Z][a-z]{2}-\d{2}"),
    # branches: only in a branch context ("Ilala branch", "Ilala on", "tawi la Ilala")
    ("BRANCH", r"\b(?:" + BRANCH_NAMES + r")\b(?= branch| on|\s+tarehe)|(?<=tawi la )(?:" + BRANCH_NAMES + r")\b"),
    # companies: up to 4 capitalised words followed by Ltd, Company Limited, ...
    ("ORG", r"(?:[A-Z][\w&']*\s){1,4}(?:Ltd|Co\. Ltd|Company Limited|Group Ltd|Enterprises)\b"),
]
print(len(RULES), "rules")

To use the rules inside spaCy, wrap them in a **pipeline component**: a function that receives each document after the
model has done its work, and can change its entities. This one:

1. finds every rule match and turns it into a spaCy span,
2. drops any of the model's own guesses that overlap a rule match (where a rule and the model disagree, the rule wins,
   because for these patterns a rule is far more reliable),
3. sets the document's entities to the rule matches plus the model's remaining guesses.

It reads `RULES` every time it runs, so you can add rules later without rebuilding anything.

Some details in the code:

- `@Language.component("fund_rules")` registers the function with spaCy under the name `"fund_rules"`, so it can be
  added to a pipeline by name;
- `re.finditer` is like `findall`, but also gives each match's start and end position;
- spaCy works in **tokens**, while the regular expressions work in characters. `doc.char_span(...)` converts character
  positions into a span of whole tokens; `alignment_mode="expand"` widens the span to whole tokens if a match starts or
  ends in the middle of one;
- `span.start` and `span.end` are token positions, so `covered` collects every token position used by a rule.

In [ ]:
@Language.component("fund_rules")
def fund_rules(doc):
    # 1. Every rule match, as a spaCy span. char_span turns character positions into whole tokens.
    rule_spans = []
    for label, pattern in RULES:
        for match in re.finditer(pattern, doc.text):
            span = doc.char_span(match.start(), match.end(), label=label, alignment_mode="expand")
            if span is not None:
                rule_spans.append(span)

    # 2. The positions of all tokens covered by a rule match...
    covered = set()
    for span in rule_spans:
        for token_position in range(span.start, span.end):
            covered.add(token_position)

    # ...and the model's entities that don't touch any of them.
    kept = []
    for ent in doc.ents:
        overlaps = False
        for token_position in range(ent.start, ent.end):
            if token_position in covered:
                overlaps = True
        if not overlaps:
            kept.append(ent)

    # 3. filter_spans keeps the longest span wherever two still overlap.
    doc.ents = filter_spans(rule_spans + kept)
    return doc

One more detail before building the pipeline. spaCy has a built-in exception that keeps *"m."* together as one token
(for abbreviations like *a.m.*), so in *"Tsh 2.38m."* the full stop would be glued to the amount and the MONEY rule
couldn't line up with the tokens. Remove that exception.

Then load a fresh copy of the model (so the original `nlp` stays unchanged for comparison) and add your component
**after** the model's own entity recogniser (`"ner"`), so it can correct the model's guesses.

In [ ]:
nlp_rules = spacy.load("en_core_web_sm")

# The tokenizer's special cases are a dictionary. Copy it, remove "m.", and put it back.
special_cases = dict(nlp_rules.tokenizer.rules)
special_cases.pop("m.", None)
nlp_rules.tokenizer.rules = special_cases

# Add our component after the model's own entity recogniser ("ner").
nlp_rules.add_pipe("fund_rules", after="ner")
print(nlp_rules.pipe_names)

Measure the new pipeline the same way, and put its F1 scores next to the standard model's:

In [ ]:
with_rules = evaluate(nlp_rules)

f1_table = pd.DataFrame({"standard model": baseline["f1"], "with rules": with_rules["f1"]})
f1_table.round(3)

The pattern-based types jump to near-perfect. Look at the regular expressions again, because each carries a piece of
local knowledge:

- `\d{3} ?\d{3} ?\d{3}` accepts member numbers written as `104837265` **or** `104 837 265`, as officers do.
- `\d[\d,]*/=` catches amounts written the Tanzanian way, `1,450,000/=`.
- The small tokenizer fix matters more than it looks. Tokenization quirks like this cause many "why doesn't my rule
  match?" puzzles; spaCy's `nlp.tokenizer.explain(text)` shows you exactly how a string was split.
- The BRANCH pattern only fires when a branch name appears in a branch context (*"Ilala branch"*, *"tawi la Ilala"*,
  *"at Kinondoni on"*), so a surname like *Moshi* in a person's name isn't mistaken for the Moshi branch.

> **Where we are.** With rules added after the model, MEMBER_NO, CLAIM_REF, BRANCH, DATE and MONEY are near-perfect.
> PERSON is still weak, because names don't follow a fixed format.

## 5. People's names

PERSON is the hard one. Names don't follow a fixed pattern, and a model trained on English news has seen few
Tanzanian names. But notes follow habits: names come after *Member*, *Mwanachama*, *Pensioner*, *for* or *to*, and
are usually followed by a member number in brackets, or by *", member no."*. Rules can use that **context**.

`(?<=Member )` means "preceded by *Member* " and `(?= \()` means "followed by a space and a bracket". Neither part is
included in the match itself.

The `NAME` pattern describes what a name looks like:

- `[A-Z]\.` an initial such as *N.*, **or** `[A-Z][a-z]+` a capitalised word such as *Neema*;
- then a space and a capitalised surname, which may contain an apostrophe (`[a-z']+`);
- `(?:-[A-Z][a-z]+)?` optionally a second, hyphenated part, as in *Mushi-Lyimo*.

Each rule then combines `NAME` with one of the contexts.

In [ ]:
# A name: an initial ("N.") or a capitalised word, then a capitalised surname (optionally double-barrelled).
NAME = r"(?:[A-Z]\.|[A-Z][a-z]+) [A-Z][a-z']+(?:-[A-Z][a-z]+)?"

PERSON_RULES = [
    ("PERSON", r"(?<=Member )" + NAME),
    ("PERSON", r"(?<=Mwanachama )" + NAME),
    ("PERSON", r"(?<=Pensioner )" + NAME),
    ("PERSON", NAME + r"(?= \(\d)"),                    # followed by "(" and a digit
    ("PERSON", NAME + r"(?=, member no\.)"),
    ("PERSON", r"^" + NAME + r"(?= reports)"),          # at the start of the note, before "reports"
    ("PERSON", r"(?<= for )" + NAME + r"(?= put on hold)"),
    ("PERSON", r"(?<= to )" + NAME + r"(?= \()"),
]

# Add them to the rule list. The component picks them up the next time it runs.
RULES = RULES + PERSON_RULES
print(len(RULES), "rules")

Measure again and add a third column to the comparison:

In [ ]:
with_person_rules = evaluate(nlp_rules)

f1_table["with rules + name context"] = with_person_rules["f1"]
f1_table.round(3)

And look at one note with the full pipeline:

In [ ]:
display(HTML(displacy.render(nlp_rules(notes["text"].iloc[4]), style="ent", jupyter=False)))

**Your turn (1).** Find the notes where PERSON is still wrong. Compare predicted and gold spans for PERSON only, print
the first ten mistakes, and see whether a pattern explains them. Can you fix any with one more rule? Be careful: every
rule you add can also create new false positives, so rerun the evaluation afterwards.

In [ ]:
# Your code here

A word of caution about rules. They're excellent when the text follows the habits you coded for, and brittle when it
doesn't. If officers start writing *"Mem. no"* instead of *"member no."*, recall drops quietly. Keep the evaluation set
and rerun it whenever the rules or the note templates change. On Day 10, that becomes a monitoring check.

> **Where we are.** Context rules have lifted PERSON a long way, and you've seen why rules need a standing evaluation set
> to stay trustworthy. The extraction works. Now make the results useful.

## 6. From entities to a table

Extraction is only half the job. The values have to be **normalised** before they're useful: `Tsh 2.38m`,
`2,380,000/=` and `TZS 2,380,000` are the same amount, and `05-Oct-23` and `2023-10-05` are the same date. A
database can only add up amounts or sort dates if each is stored in one standard form.

**Money.** Remove commas and `/=`, remove the currency word at the start (`^` means "at the start";
`re.IGNORECASE` matches upper or lower case), and if the amount ends in `m`, multiply by a million.

In [ ]:
def parse_money(text):
    """Turn any way of writing an amount ('Tsh 2.38m', '1,450,000/=', 'TZS 739,000') into a number."""
    text = text.replace(",", "").replace("/=", "").strip()
    text = re.sub(r"^(TZS|Tsh|shs)\s?", "", text, flags=re.IGNORECASE)    # remove the currency word
    if text.endswith("m"):
        return float(text[:-1]) * 1_000_000                               # "2.38m" = 2.38 million
    return float(text)


print(parse_money("Tsh 2.38m"), parse_money("1,450,000/="), parse_money("TZS 739,000"))

**Dates.** `pd.to_datetime` reads most formats on its own. Two things to tell it:

- `dayfirst=True`: in Tanzania 12/03/2025 means 12 March, not 3 December (the American reading);
- the `05-Oct-23` style needs an explicit format: `%d` day, `%b` short month name, `%y` two-digit year.

`errors="coerce"` returns an empty date (`NaT`, "not a time") instead of crashing if a date can't be read.

In [ ]:
def parse_date(text):
    """Turn any of the notes' date formats into a real date (day first, as written in Tanzania)."""
    if re.fullmatch(r"\d{2}-[A-Z][a-z]{2}-\d{2}", text):                 # 05-Oct-23
        return pd.to_datetime(text, format="%d-%b-%y", errors="coerce")
    return pd.to_datetime(text, dayfirst=True, errors="coerce")


print(parse_date("05-Oct-23"), parse_date("12/03/2025"), parse_date("5 March 2025"))

Now turn every note into one row of a table. For each note:

1. run the full pipeline and group the entities it found by type;
2. take the first member number (with its spaces removed: `\D` means "any character that isn't a digit"), the first
   person, claim reference, branch and employer, or `None` if there isn't one;
3. take the **earliest** date mentioned, and the **total** of all the amounts mentioned.

`x if found[...] else None` means "use x if the list isn't empty, otherwise None".

In [ ]:
rows = []

for note_id, text in zip(notes["note_id"], notes["text"]):
    doc = nlp_rules(text)

    # Collect the entity texts by type for this note.
    found = {}
    for label in LABELS:
        found[label] = []
    for ent in doc.ents:
        if ent.label_ in found:
            found[ent.label_].append(ent.text)

    row = {"note_id": note_id}

    # The first of each, or None if there isn't one. Member numbers lose their spaces.
    row["member_no"] = re.sub(r"\D", "", found["MEMBER_NO"][0]) if found["MEMBER_NO"] else None
    row["person"] = found["PERSON"][0] if found["PERSON"] else None
    row["claim_ref"] = found["CLAIM_REF"][0] if found["CLAIM_REF"] else None
    row["branch"] = found["BRANCH"][0] if found["BRANCH"] else None
    row["employer"] = found["ORG"][0] if found["ORG"] else None

    # The earliest date mentioned, and the total of all amounts mentioned.
    dates = []
    for d in found["DATE"]:
        dates.append(parse_date(d))
    row["first_date"] = min(dates) if dates else pd.NaT

    total = 0
    for m in found["MONEY"]:
        total = total + parse_money(m)
    row["amount_tzs"] = total if total > 0 else None

    rows.append(row)

structured = pd.DataFrame(rows)
structured.head(10)

How often was each field found? `notna()` is True where a value exists; its mean is the share of notes with that field.

In [ ]:
# How often was each field found?
structured.notna().mean().round(2)

And now a question you couldn't answer from free text: the total amounts mentioned in notes, by branch.

In [ ]:
# Total amounts mentioned, by branch, in TZS millions.
by_branch = structured.groupby("branch")["amount_tzs"].sum() / 1e6
by_branch.round(1).sort_values(ascending=False).head(8)

From 900 free-text notes to a table you can join to the member register on `member_no`, filter by date, and sum by
branch. That's the practical payoff of NER.

> **Where we are.** 900 free-text notes have become a table, with one standard form for every amount and date, ready to
> join to the member register. The same extraction can also be used to **hide** information.

### Theory: personal data, and why redaction is only one layer

**Personal data** is any information that identifies a person, directly (a name, a member number, a national ID) or
indirectly (an unusual combination of age, branch and benefit). Tanzania's Personal Data Protection Act, 2022 sets rules
on how it may be collected, used and shared.

**Redaction** replaces the identifying parts with placeholders: *"Member [NAME] ([MEMBER_NO]) visited Ilala branch..."*.
It's a useful safeguard, but not a guarantee:

- it only removes what the extraction **finds**, so its leak rate equals the extraction's misses;
- the remaining text can still identify someone (*"the only pensioner aged 104 at Moshi branch"*).

So it's one layer among several: deciding what really needs to leave the Fund, contracts and access controls for anyone
who receives it, and on Day 7, running models locally so the data doesn't leave at all.

## 7. Redaction: sharing text without sharing people

The same extraction lets you **remove** personal data. Suppose you want to send notes to an outside vendor, or to an
LLM (large language model) hosted outside the Fund (Day 7 and Day 9 will raise exactly this question). Replace the
entities that identify a person before the text leaves.

`REDACT` says which entity types to hide, and what to put in their place. The function replaces each such entity with
its placeholder. It works from the **end** of the text backwards: if you replaced the first name first, the text would
get shorter or longer and every later entity's character positions would be wrong.

`text[:a] + new + text[b:]` builds a new string: everything before position `a`, then the replacement, then everything
from position `b` on.

In [ ]:
REDACT = {"PERSON": "[NAME]", "MEMBER_NO": "[MEMBER_NO]", "CLAIM_REF": "[CLAIM_REF]"}


def redact(text):
    """Replace names, member numbers and claim references with placeholders."""
    doc = nlp_rules(text)
    redacted = text
    # Work from the END of the text backwards: replacing a later entity doesn't move the earlier ones.
    for ent in reversed(doc.ents):
        if ent.label_ in REDACT:
            redacted = redacted[:ent.start_char] + REDACT[ent.label_] + redacted[ent.end_char:]
    return redacted


for text in notes["text"].head(4):
    print(redact(text))
    print()

Redaction is only as good as the extraction behind it. Wherever PERSON recall is below 100%, some names slip through.
For anything leaving the Fund, redaction is a safeguard, not a guarantee: combine it with a contract, access controls,
and, where the data is sensitive enough, the rule that it doesn't leave at all. The Personal Data Protection Act,
2022 applies whatever tool you use.

**Your turn (2).** Measure it. For every note, check whether any gold PERSON or MEMBER_NO text still appears in the
redacted version. What share of notes leak at least one identifier?

In [ ]:
# Your code here

> **Where we are.** You can redact names, member numbers and claim references from any note, and you've seen why that's a
> safeguard rather than a guarantee.

## 8. Wrap-up

| Approach | Use it for | Weak spot |
|---|---|---|
| Pre-trained statistical NER | General names, organisations, dates | Local formats and names it has never seen |
| Rules and patterns | Anything with a fixed format: IDs, references, amounts, dates | Brittle when writing habits change |
| Context rules | Names in predictable positions | Only as good as the note-writing habits |
| Training your own NER | Fuzzy entities, at scale | Needs hundreds of annotated examples |

Whatever you use, keep the annotated evaluation set. It's how you know the extraction still works next year.

---

## Looking back, looking ahead

**What you did in this lab.** You measured spaCy's English model strictly against a hand-made answer key, added rules for
everything with a fixed format, added context rules for names, turned 900 notes into a table with normalised amounts and
dates, and used the extraction to redact personal details.

**What you did today.** Day 6 covered the three shapes of NLP task on what members and officers write: classifying
messages with classical methods (Lab 1) and a transformer (Lab 2), and labelling the facts inside them (Lab 3).

**The ideas to carry forward.**

- Measure extraction strictly: right span and right type.
- Combine a statistical model with rules that carry local knowledge.
- Normalise extracted values before anyone adds them up.
- Keep an evaluation set and rerun it whenever rules or writing habits change.
- Redaction is a safeguard, not a guarantee.

**Tomorrow (Day 7).** Large language models: one model that can summarise, classify, extract and answer, from a written
instruction, running on your own hardware.

## Suggested answers

**(1)** PERSON mistakes. For each note, compare the PERSON spans found with the PERSON spans in the answer key. Where
they differ, record the names that were missed and the text wrongly called a person:

In [ ]:
mistakes = []

for text, gold in zip(notes["text"], notes["gold"]):
    # PERSON spans the pipeline found.
    predicted = set()
    for ent in nlp_rules(text).ents:
        if ent.label_ == "PERSON":
            predicted.add((ent.start_char, ent.end_char))

    # PERSON spans in the answer key.
    truth = set()
    for g in gold:
        if g["label"] == "PERSON":
            truth.add((g["start"], g["end"]))

    if predicted != truth:
        missed = []
        for start, end in truth - predicted:
            missed.append(text[start:end])          # the text of each missed name
        wrong = []
        for start, end in predicted - truth:
            wrong.append(text[start:end])           # the text wrongly called a person
        mistakes.append({"text": text, "missed": missed, "wrong": wrong})

print(len(mistakes), "notes with a PERSON mistake")
pd.DataFrame(mistakes).head(10)

Almost all the remaining mistakes are **employer names without "Ltd"** (*Mlima Civil Works*, *Mto Guards*) that the
statistical model has labelled as people. The notes put employers in predictable places too: *"employer X has"*,
*"visit to X on"*, *"letter from X dated"*. Context rules for ORG fix them, and fixing ORG removes the false PERSON:

In [ ]:
COMPANY = r"(?:[A-Z][\w&']*\s?){1,4}"      # up to four capitalised words

ORG_CONTEXT_RULES = [
    ("ORG", r"(?<=employer )" + COMPANY + r"(?= has)"),
    ("ORG", r"(?<=visit to )" + COMPANY + r"(?= on)"),
    ("ORG", r"(?<=letter from )" + COMPANY + r"(?= dated)"),
]
RULES = RULES + ORG_CONTEXT_RULES

evaluate(nlp_rules)["f1"].round(3)

**(2)** Leakage after redaction. For each note, redact it, then check whether the text of any gold PERSON or MEMBER_NO
entity can still be found in the redacted version (`in` checks whether one string appears inside another):

In [ ]:
leaks = 0

for text, gold in zip(notes["text"], notes["gold"]):
    redacted = redact(text)
    leaked = False
    for g in gold:
        if g["label"] in ["PERSON", "MEMBER_NO"] and g["text"] in redacted:
            leaked = True
    if leaked:
        leaks += 1

print(f"{leaks} of {len(notes)} notes ({leaks / len(notes):.1%}) still contain a name or member number after redaction")

On these notes nothing leaks, because they follow a handful of writing habits the rules were built around. Real notes
are messier. Before trusting redaction on real data, hand-annotate a sample of a few hundred real notes and run exactly
this check on them.